# TaylorMade product-gallery image puller

This workflow captures every full product-gallery image—such as the four SIM2 Max views—rather than only image resources that happen to load first. It supports either a TaylorMade category/listing link (process every product card on that page) or one product link.

1. Paste the listing or product link into `PRODUCT_URL` and run the first cell.
2. In the opened browser tab, run the DevTools script from the next cell. It opens one reusable product tab, visits each product item, scrolls its gallery to trigger lazy images, walks its visible product options, and copies every gallery URL.
3. Paste the copied list into `IMAGE_URLS_TEXT` in the final cell to download the files.

The script only changes visible product-option controls; it does not add anything to the cart. Keep the normal browser window open while it runs, and allow its one popup/tab if the browser asks. Use the downloaded images only as permitted by TaylorMade's terms and the applicable image rights.

In [ ]:
from urllib.parse import urlparse
import webbrowser


# Paste a TaylorMade category/listing URL or a specific product URL here.
PRODUCT_URL = "https://www.taylormadegolf.com/PASTE-PAGE-PATH-HERE"
OPEN_PRODUCT_PAGE = True


def validate_taylormade_url(url):
    parsed = urlparse(url.strip())
    host = (parsed.hostname or "").lower()
    if parsed.scheme not in {"http", "https"} or not parsed.path or parsed.path == "/":
        raise ValueError("Set PRODUCT_URL to a full TaylorMade category or product URL.")
    if host != "taylormadegolf.com" and not host.endswith(".taylormadegolf.com"):
        raise ValueError("PRODUCT_URL must use taylormadegolf.com.")
    if "PASTE-PAGE-PATH-HERE" in parsed.path:
        raise ValueError("Replace PASTE-PAGE-PATH-HERE with the path from your TaylorMade link.")
    return url.strip()


PRODUCT_URL = validate_taylormade_url(PRODUCT_URL)
print(f"TaylorMade page: {PRODUCT_URL}")
if OPEN_PRODUCT_PAGE:
    webbrowser.open_new_tab(PRODUCT_URL)
    print("Opened the page. Run the DevTools collector in the next notebook cell.")


## DevTools gallery collector

Copy the complete block below into the **Console** of the page opened above. If Chrome or Edge shows a paste-protection prompt, follow that prompt before pasting. `MAX_PRODUCTS` limits a category run; `MAX_OPTION_COMBINATIONS_PER_PRODUCT` prevents an unexpectedly huge configurator from running without an explicit limit increase.

```js
(async () => {
  const MAX_PRODUCTS = 200;
  const MAX_OPTION_COMBINATIONS_PER_PRODUCT = 120;
  const SETTLE_MS = 850;
  const sleep = (ms) => new Promise((resolve) => setTimeout(resolve, ms));
  const imageUrls = new Set();
  const failures = [];

  const isVisible = (win, element) => {
    const style = win.getComputedStyle(element);
    return style.display !== 'none' && style.visibility !== 'hidden';
  };
  const selectorFor = (element) => {
    if (element.id) return '#' + CSS.escape(element.id);
    const parts = [];
    let current = element;
    while (current && current.nodeType === 1 && current.tagName !== 'HTML') {
      let part = current.tagName.toLowerCase();
      const siblings = Array.from(current.parentElement.children)
        .filter((sibling) => sibling.tagName === current.tagName);
      if (siblings.length > 1) part += `:nth-of-type(${siblings.indexOf(current) + 1})`;
      parts.unshift(part);
      current = current.parentElement;
    }
    return 'html > ' + parts.join(' > ');
  };
  const labelFor = (element, fallback) => {
    const container = element.closest('fieldset, [role="radiogroup"], [data-attribute], [class*="attribute" i], [class*="variation" i], [class*="swatch" i]');
    return (element.getAttribute('aria-label') || (container && container.getAttribute('aria-label')) || element.name || element.id || fallback)
      .replace(/\s+/g, ' ').trim();
  };
  const largestSrcsetUrl = (srcset) => {
    if (!srcset) return null;
    const entries = srcset.split(',').map((item) => item.trim().split(/\s+/)[0]).filter(Boolean);
    return entries.at(-1) || null;
  };

  async function loadAllListingItems() {
    let stablePasses = 0;
    let previousHeight = 0;
    for (let pass = 0; pass < 30 && stablePasses < 2; pass += 1) {
      const moreButton = Array.from(document.querySelectorAll('button'))
        .find((button) => /^(load|show) more$/i.test((button.innerText || '').trim()) && !button.disabled);
      if (moreButton) moreButton.click();
      window.scrollTo(0, document.body.scrollHeight);
      await sleep(900);
      const height = document.body.scrollHeight;
      stablePasses = height === previousHeight && !moreButton ? stablePasses + 1 : 0;
      previousHeight = height;
    }
    window.scrollTo(0, 0);
  }

  function listingProductUrls() {
    const cardSelector = [
      '[data-product-id] a[href]', '[data-pid] a[href]', '[class*="product-tile" i] a[href]',
      '[class*="product-card" i] a[href]', '[class*="product-item" i] a[href]',
      '[class*="product-grid" i] a[href]', '[class*="product-list" i] a[href]'
    ].join(', ');
    let anchors = Array.from(document.querySelectorAll(cardSelector));
    if (!anchors.length) anchors = Array.from(document.querySelectorAll("a[href*='.html'], a[href*='Product-Show']"));
    const urls = new Set();
    for (const anchor of anchors) {
      const url = new URL(anchor.href, location.href);
      const isProductPath = /\.html$/i.test(url.pathname) || /Product-Show/i.test(url.pathname + url.search);
      const card = anchor.closest('[data-product-id], [data-pid], [class*="tile" i], [class*="product-card" i], [class*="product-item" i]');
      if (url.origin === location.origin && isProductPath && (card || anchor.querySelector('img'))) urls.add(url.href);
    }
    if (urls.size > MAX_PRODUCTS) {
      throw new Error('Found ' + urls.size + ' product items. Increase MAX_PRODUCTS before collecting every item.');
    }
    return Array.from(urls);
  }

  function optionGroups(doc, win) {
    const root = doc.querySelector('main') || doc.body;
    const groups = [];
    for (const select of root.querySelectorAll('select')) {
      if (select.disabled || !isVisible(win, select)) continue;
      const options = Array.from(select.options)
        .filter((item) => !item.disabled && item.value && !/select|choose/i.test(item.textContent))
        .map((item) => ({ value: item.value, label: item.textContent.trim() }));
      if (options.length > 1) groups.push({ kind: 'select', key: selectorFor(select), label: labelFor(select, 'select'), selector: selectorFor(select), options });
    }
    const radiosByName = new Map();
    for (const radio of root.querySelectorAll('input[type="radio"]')) {
      if (radio.disabled) continue;
      const key = radio.name || selectorFor(radio.closest('fieldset, [role="radiogroup"]') || radio.parentElement);
      if (!radiosByName.has(key)) radiosByName.set(key, []);
      radiosByName.get(key).push(radio);
    }
    for (const [key, radios] of radiosByName) {
      if (radios.length < 2) continue;
      groups.push({ kind: 'click', key: `radio:${key}`, label: labelFor(radios[0], key), options: radios.map((radio) => {
        const label = radio.id && doc.querySelector(`label[for="${CSS.escape(radio.id)}"]`);
        return { selector: selectorFor(label || radio), label: (label && label.innerText) || radio.getAttribute('aria-label') || radio.value };
      }) });
    }
    const buttonGroups = new Map();
    for (const button of root.querySelectorAll('button, [role="radio"]')) {
      if (button.disabled || !isVisible(win, button)) continue;
      const container = button.closest('[role="radiogroup"], fieldset, [data-attribute], [class*="attribute" i], [class*="variation" i], [class*="swatch" i], [class*="product-option" i]');
      if (!container || container.querySelector('select, input[type="radio"]')) continue;
      const key = selectorFor(container);
      if (!buttonGroups.has(key)) buttonGroups.set(key, { container, buttons: [] });
      buttonGroups.get(key).buttons.push(button);
    }
    for (const [key, entry] of buttonGroups) {
      if (entry.buttons.length < 2 || entry.buttons.length > 40) continue;
      groups.push({ kind: 'click', key: `button:${key}`, label: labelFor(entry.container, 'option'), options: entry.buttons.map((button) => ({
        selector: selectorFor(button), label: (button.getAttribute('aria-label') || button.innerText || button.title || 'option').trim()
      })) });
    }
    const seen = new Set();
    return groups.filter((group) => !seen.has(group.key) && seen.add(group.key));
  }

  async function wakeGallery(win, doc) {
    const maxScroll = Math.max(doc.body.scrollHeight, doc.documentElement.scrollHeight);
    const step = Math.max(300, Math.floor(win.innerHeight * 0.75));
    for (let position = 0; position <= maxScroll; position += step) {
      win.scrollTo(0, position);
      await sleep(120);
    }
    win.scrollTo(0, 0);
    await sleep(250);
  }

  function collectGalleryImages(win, doc) {
    const root = doc.querySelector('main') || doc.body;
    const gallerySelector = '[data-testid*="gallery" i], [class*="gallery" i], [class*="product-image" i], [class*="pdp-image" i], [class*="image-gallery" i]';
    const galleryContainers = Array.from(root.querySelectorAll(gallerySelector));
    let images = galleryContainers.flatMap((container) => Array.from(container.querySelectorAll('img')));
    if (!images.length) {
      images = Array.from(root.querySelectorAll('img')).filter((image) => {
        const box = image.getBoundingClientRect();
        return box.width >= 160 || box.height >= 160 || image.naturalWidth >= 500 || image.naturalHeight >= 500;
      });
    }
    for (const image of images) {
      const box = image.getBoundingClientRect();
      const explicitGallerySource = image.dataset.zoomImage || image.dataset.zoom || image.dataset.largeImage || image.dataset.full;
      if (!explicitGallerySource && box.width < 120 && box.height < 120) continue;
      const sources = [
        explicitGallerySource, largestSrcsetUrl(image.getAttribute('srcset')), image.currentSrc, image.getAttribute('data-src'), image.src
      ];
      for (const source of sources.filter(Boolean)) {
        const url = new URL(source, win.location.href).href;
        if (!/\.(jpg|jpeg|png|webp|avif)(\?|$)/i.test(url)) continue;
        if (/(?:logo|icon|sprite|flag|payment)/i.test(new URL(url).pathname)) continue;
        imageUrls.add(url);
      }
    }
    for (const container of galleryContainers) {
      const background = win.getComputedStyle(container).backgroundImage;
      for (const match of background.matchAll(/url\(["']?(.*?)["']?\)/g)) {
        const url = new URL(match[1], win.location.href).href;
        if (/\.(jpg|jpeg|png|webp|avif)(\?|$)/i.test(url)) imageUrls.add(url);
      }
    }
  }

  async function processProduct(win, productUrl, number, total) {
    win.location.replace(productUrl);
    for (let attempt = 0; attempt < 40; attempt += 1) {
      await sleep(500);
      if (win.closed) throw new Error('The product tab was closed.');
      if (win.document.readyState === 'complete' && win.document.querySelector('main, body')) break;
    }
    const doc = win.document;
    const title = (doc.querySelector('h1') || {}).innerText || productUrl;
    await sleep(1200);
    await wakeGallery(win, doc);
    collectGalleryImages(win, doc);
    const groups = optionGroups(doc, win);
    const combinations = groups.reduce((all, group) =>
      all.flatMap((selection) => group.options.map((option) => [...selection, { group, option }])), [[]]
    );
    if (combinations.length > MAX_OPTION_COMBINATIONS_PER_PRODUCT) {
      throw new Error(`${title}: ${combinations.length} option combinations exceeds MAX_OPTION_COMBINATIONS_PER_PRODUCT.`);
    }
    for (let index = 0; index < combinations.length; index += 1) {
      let available = true;
      for (const { group, option } of combinations[index]) {
        if (group.kind === 'select') {
          const select = doc.querySelector(group.selector);
          if (!select) { available = false; break; }
          select.value = option.value;
          select.dispatchEvent(new win.Event('input', { bubbles: true }));
          select.dispatchEvent(new win.Event('change', { bubbles: true }));
          if (select.value !== option.value) { available = false; break; }
        } else {
          const target = doc.querySelector(option.selector);
          if (!target || target.disabled) { available = false; break; }
          target.click();
        }
        await sleep(180);
      }
      if (!available) continue;
      await sleep(SETTLE_MS);
      await wakeGallery(win, doc);
      collectGalleryImages(win, doc);
    }
    console.info(`[${number}/${total}] ${title}: ${combinations.length} option states scanned.`);
  }

  const currentIsProduct = /\.html$/i.test(location.pathname) && Boolean(document.querySelector('main h1, h1'));
  if (!currentIsProduct) await loadAllListingItems();
  const productUrls = currentIsProduct ? [location.href] : listingProductUrls();
  if (!productUrls.length) throw new Error('No product item links were found on this page. Open a product page directly or use a category page with product cards.');
  console.info(`Processing ${productUrls.length} product item(s).`);
  const worker = window.open('about:blank', 'taylormade-gallery-collector', 'popup,width=1440,height=1000');
  if (!worker) throw new Error('The collector popup was blocked. Allow one popup for this TaylorMade page and run the script again.');

  for (let index = 0; index < productUrls.length; index += 1) {
    try {
      await processProduct(worker, productUrls[index], index + 1, productUrls.length);
    } catch (error) {
      failures.push({ url: productUrls[index], reason: String(error) });
      console.warn(`Skipped ${productUrls[index]}: ${error}`);
    }
  }

  const result = Array.from(imageUrls).sort().join('\n');
  console.log(result);
  if (typeof copy === 'function') copy(result);
  else await navigator.clipboard.writeText(result);
  console.info(`Copied ${imageUrls.size} gallery image URLs from ${productUrls.length - failures.length}/${productUrls.length} product items.`);
  if (failures.length) console.table(failures);
})();
```

In [ ]:
from pathlib import Path
from urllib.parse import urlparse
import re

import requests


PROJECT_ROOT = next(
    parent
    for parent in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (parent / "requirements.txt").is_file()
)
# Change this classification when importing another kind of club.
OUTPUT_SUBDIRECTORY = Path("iron") / "irons"
OUTPUT_DIR = PROJECT_ROOT / "assets" / "club-reference-images" / OUTPUT_SUBDIRECTORY
IMAGE_URL_PATTERN = re.compile(r"\.(?:jpg|jpeg|png|webp|avif)(?:\?.*)?$", re.IGNORECASE)


# Paste the DevTools collector's copied URLs here: one URL per line.
IMAGE_URLS_TEXT = """
"""

IMAGE_URLS = sorted(
    {
        url
        for line in IMAGE_URLS_TEXT.splitlines()
        if (url := line.strip()) and IMAGE_URL_PATTERN.search(url)
    }
)
if not IMAGE_URLS:
    raise RuntimeError(
        "No gallery image URLs were found. Paste the DevTools collector output into IMAGE_URLS_TEXT, then run this cell again."
    )

print(f"Found {len(IMAGE_URLS)} gallery image URLs.")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

session = requests.Session()
session.headers.update(
    {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 Chrome/142.0 Safari/537.36"
        ),
        "Referer": "https://www.taylormadegolf.com/",
    }
)

for index, image_url in enumerate(IMAGE_URLS, start=1):
    filename = Path(urlparse(image_url).path).name or f"image_{index}.jpg"
    destination = OUTPUT_DIR / filename
    if destination.exists():
        destination = OUTPUT_DIR / f"{Path(filename).stem}_{index}{Path(filename).suffix or '.jpg'}"

    try:
        response = session.get(image_url, timeout=30)
        response.raise_for_status()
        destination.write_bytes(response.content)
        print(f"Downloaded: {destination}")
    except requests.RequestException as error:
        print(f"Failed: {image_url}\nReason: {error}")
